# Chapter 10 - Part 1: Molecular features for machine learning

## Learning objectives

By the end of this notebook, you should be able to:

1. Separate a measured target, a molecular representation, and a learned prediction.
2. Audit SMILES before computing named descriptors and modern RDKit fingerprints.
3. Explain bit versus count fingerprints, Morgan radius, hash collisions, and Tanimoto similarity.
4. Test atom-order invariance and identify information lost about stereochemistry, protonation, and conditions.
5. Design a data and evaluation protocol before fitting a model in [Part 2](Chapter10_Part2.ipynb).

**Run from the repository root using the environment in [Readme.md](Readme.md).** All examples are small, offline calculations. The 12 structures below form an illustrative collection; **they have no measured property labels**. Descriptor values calculated here are not experimental measurements. Figures and tables are written under `outputs/chapter10_part1/`.

### Start here: turn a molecule into a row of numbers

A computer model cannot learn directly from the word “phenol.” We must specify its structure, choose which information to retain, and turn that information into numerical inputs. Each molecule becomes one **row**; each descriptor or fingerprint position becomes one **feature column**.

| Term | Example in this lesson | What it means |
|---|---|---|
| Structure | Phenol's atoms, bonds, and specified charge | The molecular object being represented |
| Descriptor | Molecular mass or a donor count | A named summary calculated from that structure |
| Fingerprint | Bits encoding molecular neighborhoods | A compact structural representation; one bit can have more than one origin |
| Target or label | A measured property we would like to predict | Absent from this illustrative collection |
| Prediction | A fitted model's estimate of a target | Requires labeled training data and an evaluation protocol |

The vector $\mathbf x$ is simply a molecule's ordered list of features. In Part 2, $y$ will be an experimental target and $\hat y$ its prediction. **First pass:** follow the molecular drawings, named descriptor table, and analogue-selection example. Generator settings and collision tests explain how a representation can lose information.

## 1. What is being learned?

A supervised model learns a mapping $\hat y=f_\theta(\mathbf{x})$ from features $\mathbf{x}$ to a target $y$, using training examples to choose parameters $\theta$.

| Learning task | Chemical example | What must be checked |
|---|---|---|
| Regression | Predict a measured distribution coefficient, log D at a specified pH | Endpoint, units, assay conditions, and experimental uncertainty |
| Classification | Predict a dataset's binary assay label | Meaning of both classes, class prevalence, and the decision threshold |
| Unsupervised learning | Explore structural groups without property labels | A visible cluster need not correspond to a biological mechanism |
| Reinforcement learning | Choose sequential actions using a specified reward | A high model reward need not imply a synthesizable or useful molecule |

More data help only if their labels, coverage, and relationship to the intended prediction problem are suitable. A model can learn assay artifacts, duplicate compounds, or a series identifier instead of transferable chemistry. Better fit to the training set is not evidence of better predictions on new chemistry.

**Data sources are not interchangeable.** PubChem and ChEMBL contain structures and assay records; the PDB contains experimentally determined macromolecular structures; the Open Reaction Database contains structured reaction records. MoleculeNet packages several property-prediction benchmarks. Record the original source, license, structure-processing choices, endpoint definition, conditions, and split alongside any derived table. A SMILES string alone cannot specify all of these. See the [MoleculeNet paper](https://doi.org/10.1039/C7SC02664A) and [Open Reaction Database paper](https://doi.org/10.1021/jacs.1c09820).

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import Descriptors, Fragments, MACCSkeys, rdFingerprintGenerator
from rdkit.Chem.Draw import rdMolDraw2D

OUT = Path('outputs/chapter10_part1')
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
print('RDKit:', rdBase.rdkitVersion, '| NumPy:', np.__version__)

## 2. Structures first: preserve the row-to-molecule link

Parsing includes RDKit's valence and aromaticity checks. Successful parsing does **not** establish that a record is the intended compound, the correct tautomer, or the dominant protonation state at an assay pH. A dot separates disconnected fragments and often indicates a salt or mixture. Salt removal, charge neutralization, and tautomer standardization need explicit policies; they can alter the scientific meaning of a record.

Never catch a failed parse and silently continue with the original label array: features and labels can then describe different compounds. Retain a stable row identifier and an audit result. Here all examples are deliberately valid single-component structures.

In [ ]:
records = [
    ('ethanol', 'CCO'),
    ('dimethyl ether', 'COC'),
    ('benzene', 'c1ccccc1'),
    ('phenol', 'Oc1ccccc1'),
    ('aniline', 'Nc1ccccc1'),
    ('acetic acid', 'CC(=O)O'),
    ('acetate', 'CC(=O)[O-]'),
    ('lactic acid A', 'C[C@H](O)C(=O)O'),
    ('lactic acid B', 'C[C@@H](O)C(=O)O'),
    ('4-nitrophenol', 'Oc1ccc([N+](=O)[O-])cc1'),
    ('aspirin', 'CC(=O)Oc1ccccc1C(=O)O'),
    ('caffeine', 'Cn1c(=O)c2c(ncn2C)n(C)c1=O'),
]
structures = pd.DataFrame(records, columns=['name', 'input_smiles'])
structures.index.name = 'row_id'
mols = []
for row_id, row in structures.iterrows():
    mol = Chem.MolFromSmiles(row.input_smiles)
    if mol is None or len(Chem.GetMolFrags(mol)) != 1:
        raise ValueError(f'Invalid or disconnected teaching structure at row {row_id}')
    mols.append(mol)
structures['canonical_smiles'] = [Chem.MolToSmiles(m, isomericSmiles=True) for m in mols]
structures['formal_charge'] = [Chem.GetFormalCharge(m) for m in mols]
structures['heavy_atoms'] = [m.GetNumHeavyAtoms() for m in mols]
assert len(mols) == len(structures) == 12
display(structures)

In [ ]:
# Explicit Cairo bytes work in both a notebook kernel and a Python script.
drawer = rdMolDraw2D.MolDraw2DCairo(1000, 750, 250, 250)
drawer.DrawMolecules(mols, legends=structures['name'].tolist())
drawer.FinishDrawing()
structure_png = drawer.GetDrawingText()
(OUT / 'teaching_structures.png').write_bytes(structure_png)
display(Image(data=structure_png))

## 3. Named descriptors and fragments

A descriptor summarizes some aspect of a structure. Select and name features deliberately instead of relying on the order of a private descriptor list, which can change between releases.

| Feature | Interpretation |
|---|---|
| `MolWt` | Average molecular mass in daltons, numerically equal to molar mass in g/mol |
| `HBD`, `HBA` | Rule-based hydrogen-bond donor and acceptor counts; neither is a direct binding measurement |
| `TPSA` | Fragment-based topological polar surface area in square angstroms; not a computed 3D solvent-accessible area |
| `FractionCSP3` | Fraction of carbon atoms assigned sp3 hybridization |
| `MolLogP` | RDKit's Wildman-Crippen estimate of log P; not measured log D at an assay pH |
| Ring and fragment counts | Counts under RDKit's ring and SMARTS definitions; definitions matter, especially for fused rings |

Log P describes partitioning of a specified neutral species; log D includes the distribution of all relevant ionization states at a specified pH. A graph descriptor does not perform an acid-base equilibrium calculation. See the [RDKit descriptor API](https://www.rdkit.org/docs/source/rdkit.Chem.Descriptors.html), [Crippen API](https://www.rdkit.org/docs/source/rdkit.Chem.Crippen.html), and [Wildman-Crippen paper](https://doi.org/10.1021/ci990307l).

In [ ]:
descriptor_functions = {
    'MolWt': Descriptors.MolWt,
    'HBD': Descriptors.NumHDonors,
    'HBA': Descriptors.NumHAcceptors,
    'heteroatoms': Descriptors.NumHeteroatoms,
    'rings': Descriptors.RingCount,
    'aromatic_rings': Descriptors.NumAromaticRings,
    'aliphatic_rings': Descriptors.NumAliphaticRings,
    'saturated_rings': Descriptors.NumSaturatedRings,
    'FractionCSP3': Descriptors.FractionCSP3,
    'TPSA': Descriptors.TPSA,
    'MolLogP': Descriptors.MolLogP,
}
descriptors = pd.DataFrame(
    [{name: function(mol) for name, function in descriptor_functions.items()} for mol in mols],
    index=structures['name'],
)
assert np.isfinite(descriptors.to_numpy()).all()
descriptors.to_csv(OUT / 'descriptors.csv')
display(descriptors.round(3))

In [ ]:
nitrophenol = mols[9]
fragment_counts = pd.Series({
    'aromatic OH groups': Fragments.fr_Ar_OH(nitrophenol),
    'nitro groups': Fragments.fr_nitro(nitrophenol),
    'total formal charge': Chem.GetFormalCharge(nitrophenol),
})
assert fragment_counts.tolist() == [1, 1, 0]
display(fragment_counts.to_frame('4-nitrophenol'))
print('The nitro group has internal formal charges, while this molecule has net charge zero.')

### Representation can discard information

The descriptor vector can be identical for distinct structures. Two enantiomers have the same values for the descriptors selected above; a model receiving only these values must give them the same prediction. That may be appropriate for an achiral bulk property but cannot represent different responses in a chiral environment. Conversely, different SMILES traversals of **the same** molecular graph should not change graph-based features.

In [ ]:
np.testing.assert_allclose(descriptors.loc['lactic acid A'], descriptors.loc['lactic acid B'])
ethanol_a = Chem.MolFromSmiles('CCO')
ethanol_b = Chem.MolFromSmiles('OCC')
assert Chem.MolToSmiles(ethanol_a) == Chem.MolToSmiles(ethanol_b)
assert structures.loc[0, 'canonical_smiles'] != structures.loc[1, 'canonical_smiles']
assert np.isclose(descriptors.loc['ethanol', 'MolWt'], descriptors.loc['dimethyl ether', 'MolWt'])
print('Equal mass does not identify a structure; these enantiomers share the selected descriptor vector.')

## 4. Fingerprints: substructures as features

Use RDKit's current **fingerprint generators**, which share methods for bit and count output. Parameters are part of the representation and must be saved with a model. The table describes the settings used here, not every variant of each family.

| Fingerprint | What is encoded here |
|---|---|
| Morgan, radius 2 | Atom-centered neighborhoods out to two bonds; analogous to the diameter-4 convention in ECFP4, with implementation-specific invariants |
| RDKit topological | Paths and branched subgraphs up to seven bonds |
| Atom pair, 2D | Atom types and shortest-path distances in bonds, including nonadjacent pairs; not Euclidean distances |
| Topological torsion | Sequences of four bonded atoms; a graph feature, not a numerical dihedral angle |
| MACCS | RDKit's implementation of the public 166 structural keys; stored in 167 positions, with index 0 unused |

For the generator examples, `countSimulation=False` makes a bit indicate occurrence of a hashed feature. A **count** vector instead accumulates feature multiplicity. Different features may hash to the same index in either fixed-length representation. MACCS uses predefined keys with implementation-specific definitions. Sources: [RDKit fingerprint guide](https://www.rdkit.org/docs/GettingStartedInPython.html#fingerprinting-and-molecular-similarity), [generator API](https://www.rdkit.org/docs/source/rdkit.Chem.rdFingerprintGenerator.html), [MACCS definitions](https://www.rdkit.org/docs/source/rdkit.Chem.MACCSkeys.html), and the [ECFP paper](https://doi.org/10.1021/ci100050t).

In [ ]:
N_BITS = 1024
morgan = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=N_BITS, includeChirality=True, countSimulation=False,
)
generators = {
    'Morgan r=2': morgan,
    'RDKit paths': rdFingerprintGenerator.GetRDKitFPGenerator(
        maxPath=7, fpSize=N_BITS, countSimulation=False),
    'atom pairs (2D)': rdFingerprintGenerator.GetAtomPairGenerator(
        use2D=True, fpSize=N_BITS, countSimulation=False),
    'topological torsions': rdFingerprintGenerator.GetTopologicalTorsionGenerator(
        torsionAtomCount=4, fpSize=N_BITS, countSimulation=False),
}
fingerprint_summary = []
for name, generator in generators.items():
    bits = generator.GetFingerprint(nitrophenol)
    counts = generator.GetCountFingerprint(nitrophenol)
    fingerprint_summary.append({
        'family': name, 'length': bits.GetNumBits(), 'on_bits': bits.GetNumOnBits(),
        'count_total': sum(counts.GetNonzeroElements().values()),
    })
maccs = MACCSkeys.GenMACCSKeys(nitrophenol)
assert maccs.GetNumBits() == 167 and maccs[0] == 0
display(pd.DataFrame(fingerprint_summary).set_index('family'))
print('MACCS length:', maccs.GetNumBits(), '| on bits:', maccs.GetNumOnBits())

In [ ]:
fingerprints = [morgan.GetFingerprint(m) for m in mols]
X_bits = np.stack([morgan.GetFingerprintAsNumPy(m) for m in mols])
X_counts = np.stack([morgan.GetCountFingerprintAsNumPy(m) for m in mols])
assert X_bits.shape == X_counts.shape == (12, N_BITS)
assert set(np.unique(X_bits)) <= {0, 1}
assert np.any(X_counts > 1)
assert morgan.GetFingerprint(ethanol_a) == morgan.GetFingerprint(ethanol_b)
print('Feature matrix: rows are molecules, columns are fingerprint positions:', X_bits.shape)
print('Benzene: occupied bit positions =', int(X_bits[2].sum()),
      '; summed counts =', int(X_counts[2].sum()))

### Chirality is a choice, not an automatic guarantee

The next check explicitly switches chirality encoding. Unspecified stereochemistry stays unspecified: enabling the option cannot reconstruct missing experimental information. Protonation and tautomer changes can also change a fingerprint without changing the compound identifier used in a database. Hash collisions mean even a chirality-enabled bit vector is not a unique chemical identifier.

In [ ]:
without_chirality = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=N_BITS, includeChirality=False)
assert without_chirality.GetFingerprint(mols[7]) == without_chirality.GetFingerprint(mols[8])
assert fingerprints[7] != fingerprints[8]
assert fingerprints[5] != fingerprints[6]
print('Enantiomers identical without chirality:', True)
print('Enantiomers distinct with our chirality-enabled settings:', fingerprints[7] != fingerprints[8])
print('Acetic acid and acetate differ:', fingerprints[5] != fingerprints[6])

### Folding collisions: inspect a deliberately small fingerprint

An unfolded Morgan count fingerprint uses a large integer identifier space. It avoids the additional folding into a short vector but can still have identifier collisions. Below, two different unfolded identifiers can map to the same index after folding modulo `fpSize`. This is a demonstration of lost resolution, **not** a test of predictive accuracy. A larger fingerprint costs memory and does not guarantee a better model.

In [ ]:
unfolded = morgan.GetSparseCountFingerprint(nitrophenol).GetNonzeroElements()
collision_rows = []
for size in [16, 64, 256, 1024]:
    occupied = len({identifier % size for identifier in unfolded})
    collision_rows.append({'bits': size, 'unfolded_identifiers': len(unfolded),
                           'occupied_positions': occupied,
                           'extra_identifiers_sharing_positions': len(unfolded) - occupied})
collision_table = pd.DataFrame(collision_rows)
assert collision_table.iloc[0]['extra_identifiers_sharing_positions'] > 0
display(collision_table)

## 5. Similarity is conditional on a representation

For two nonempty binary fingerprints with $a$ and $b$ set bits and $c$ shared set bits, the Tanimoto coefficient is

$$T(A,B)=\frac{c}{a+b-c}.$$

This is an overlap score for the chosen representation. It is not a probability of equal biological activity, and no universal similarity threshold defines a model's applicability. Similar compounds can have sharply different activities. Different fingerprint families or protonation policies can change the ranking. The equation here applies to binary vectors; count-vector similarity needs its own defined convention.

In [ ]:
similarity = np.array([DataStructs.BulkTanimotoSimilarity(fp, fingerprints) for fp in fingerprints])
np.testing.assert_allclose(similarity, similarity.T)
np.testing.assert_allclose(np.diag(similarity), 1.0)
a, b = X_bits[3].astype(bool), X_bits[9].astype(bool)
manual = np.count_nonzero(a & b) / np.count_nonzero(a | b)
assert np.isclose(manual, similarity[3, 9])
fig, ax = plt.subplots(figsize=(9.4, 7.4), layout='constrained')
im = ax.imshow(similarity, vmin=0, vmax=1, cmap='viridis')
ax.set_xticks(range(12), structures['name'], rotation=65, ha='right')
ax.set_yticks(range(12), structures['name'])
ax.set_title('Morgan radius 2, 1024 bits, chirality enabled')
fig.colorbar(im, ax=ax, label='Binary Tanimoto similarity', shrink=0.8)
fig.savefig(OUT / 'fingerprint_similarity.png')
plt.show()
print(f'Phenol / 4-nitrophenol Tanimoto: {manual:.3f}')

### Worked research question: which molecules would you inspect as phenol analogues?

Suppose a researcher wants a small list of molecules to inspect alongside phenol before deciding what to measure. There are no activity labels in this collection, so we can organize structures but cannot name a biological “winner.”

Use two complementary views of the **same 12 structures**:

1. A descriptor map plots calculated molecular mass against TPSA. These two physical summaries are easy to read, but discard most connectivity information. No PCA, fitted scaling, or activity model is involved.
2. The already defined Morgan fingerprints rank candidates by shared local structural features. Exclude phenol itself and take the first three as a **predeclared inspection budget**, breaking any exact similarity ties by the original row identifier.

Read the axes before interpreting distance: the descriptor plot uses different physical units on its two axes. Visual proximity is not a calibrated distance or a probability of similar behavior. Descriptor definitions follow the [RDKit descriptor API](https://www.rdkit.org/docs/source/rdkit.Chem.Descriptors.html); the fingerprint and Tanimoto conventions are the ones defined above.

In [ ]:
query_index = int(structures.index[structures.name == 'phenol'][0])
candidate_indices = np.array([i for i in range(len(mols)) if i != query_index])
candidate_order = candidate_indices[np.lexsort((candidate_indices, -similarity[query_index, candidate_indices]))]
inspection_budget = 3
analogue_shortlist = structures.loc[candidate_order[:inspection_budget], ['name', 'canonical_smiles']].copy()
analogue_shortlist['Morgan_Tanimoto_to_phenol'] = similarity[query_index, candidate_order[:inspection_budget]]
assert query_index not in analogue_shortlist.index
display(analogue_shortlist)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), layout='constrained')
mass_values = descriptors.MolWt.to_numpy()
area_values = descriptors.TPSA.to_numpy()
points = axes[0].scatter(mass_values[candidate_indices], area_values[candidate_indices],
                          c=similarity[query_index, candidate_indices], cmap='viridis',
                          vmin=0, vmax=1, s=55, edgecolors='white')
axes[0].scatter([mass_values[query_index]], [area_values[query_index]], marker='*', s=180,
                color='#d18a26', edgecolors='black', zorder=4)
label_offsets = {0:(-10,8), 1:(-10,-17), 2:(0,-17), 3:(10,-15), 4:(8,9),
                 5:(-14,-17), 6:(-15,9), 7:(-30,9), 9:(-20,9), 10:(-12,-17), 11:(-30,9)}
for i, offset in label_offsets.items():
    label = 'lactic acids A/B' if i == 7 else structures.loc[i, 'name']
    axes[0].annotate(label, (mass_values[i], area_values[i]), xytext=offset,
                      textcoords='offset points', fontsize=8)
axes[0].set(xlabel='Calculated molecular mass (Da)', ylabel='Calculated TPSA (Å²)',
            title='Two descriptors reveal size and polar-area coverage', xlim=(20, 230), ylim=(-8, 80))
fig.colorbar(points, ax=axes[0], label='Morgan Tanimoto to phenol', shrink=0.75)
shown = candidate_order[:6]
axes[1].barh(np.arange(len(shown)), similarity[query_index, shown],
              color=['#28788e' if j < inspection_budget else '#aeb8bb' for j in range(len(shown))])
axes[1].set(yticks=np.arange(len(shown)), yticklabels=structures.loc[shown, 'name'],
            xlabel='Morgan Tanimoto to phenol', xlim=(0, 1), title='A structural inspection list; no measured activity')
axes[1].invert_yaxis()
for j, index in enumerate(shown):
    axes[1].text(similarity[query_index, index] + 0.015, j,
                  f'{similarity[query_index, index]:.3f}', va='center', fontsize=9)
fig.savefig(OUT / 'phenol_analogue_inspection.png')
plt.show()
analogue_shortlist.to_csv(OUT / 'phenol_analogue_inspection.csv')

**Conclusion.** The map helps check the size and calculated polar-area coverage of the proposed inspection list; the fingerprint ranking supplies a different, connectivity-sensitive view. Neither proves similar potency, permeability, solubility, or experimental suitability. The two lactic-acid stereoisomers occupy exactly the same descriptor location, illustrating how a plot can hide distinct structures.

**Next step:** inspect the actual molecular drawings, define the endpoint and conditions, and obtain suitable measurements before learning or transferring property labels. A descriptor-space map can organize a catalog even when no predictive model can yet be justified.

**Self-check:** if you add activity labels later, may you pick the fingerprint radius that best separates the final test labels? <details><summary>Answer</summary>No. Representation selection must use training/validation information. A visually convincing test-label separation would already have influenced the chosen model input.</details>

## 6. From features to a defensible experiment

1. **Define the target and prediction setting.** Predicting a new analog within a known series differs from predicting a new chemical series or a future assay batch.
2. **Audit records and split related chemistry together.** Canonical SMILES is useful for exact graph identity; it does not automatically merge tautomers, salts, or all related analogs. Conformers and randomized SMILES of one compound must remain in the same split.
3. **Separate fixed feature calculation from learned preprocessing.** Computing the same fixed descriptor formula for each structure uses no other rows. Imputation, scaling, variance filtering, PCA, and supervised feature selection learn from the data and belong inside the training pipeline, including during cross-validation.
4. **Choose representations and hyperparameters using training/validation data.** Keep an untouched test set for the final assessment. Changing settings after inspecting test scores turns the test set into another validation set.
5. **Compare a simple baseline and report limitations.** More features or a more flexible algorithm do not ensure useful prediction.

PCA finds linear directions of high feature variance after the chosen scaling; high variance does not necessarily mean high relevance to the target. t-SNE is primarily a nonlinear visualization method: global distances, cluster sizes, and apparent gaps can be misleading, and it is not a generic fitted feature transform for new molecules. Neither plot validates a predictive model. See [scikit-learn's leakage guidance](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage) and [t-SNE API notes](https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html).

In [ ]:
manifest = {
    'rdkit_version': rdBase.rdkitVersion,
    'dataset': '12 embedded illustrative structures; no measured labels',
    'descriptor_names': list(descriptor_functions),
    'fingerprint': {'family': 'Morgan', 'radius': 2, 'fpSize': N_BITS,
                    'includeChirality': True, 'countSimulation': False},
    'structure_policy': 'parse/sanitize supplied SMILES; no salt, charge, or tautomer transformations',
}
structures.to_csv(OUT / 'structures.csv')
np.savez_compressed(OUT / 'morgan_features.npz', bits=X_bits, counts=X_counts)
(OUT / 'feature_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Saved named features, row identifiers, settings, and figures to', OUT)

## Exercises and selected answers

1. Why can an ethanol record written as `OCC` share a model input with `CCO`, while `COC` should not be treated as a duplicate?
2. Benzene sets few Morgan bits but has a larger summed count. Explain without equating either number to the number of atoms.
3. Repeat the similarity calculation with radius 1. Before calculating, predict whether all pairwise scores must increase, decrease, or neither.
4. An assay depends on pH, but the CSV contains only canonical SMILES and a label. What information should you seek before merging it with another assay?
5. You generate ten conformers per molecule and randomly split conformers into training and test sets. What goes wrong?
6. You select the fingerprint radius with the best test-set score. Is that score still an independent estimate of generalization?

**Selected answers.** (1) `OCC` and `CCO` describe the same connectivity; `COC` is dimethyl ether, a constitutional isomer with the same formula. (2) Repeated symmetry-related environments contribute counts to shared identifiers; fingerprint generation also has rules about redundant environments. (3) Neither: changing the representation changes both shared and total features. (4) Seek endpoint definition, pH, temperature where relevant, assay protocol, units, uncertainty, and the identity/protonation/tautomer processing policy. (5) The same compound and label can occur in both sets, exaggerating transfer to new molecules; split at the compound or appropriate chemical-group level first. (6) No: the test data have influenced model selection.

**Next:** [Part 2 - regression, classification, and evaluation](Chapter10_Part2.ipynb).